In [ ]:
!pip install --upgrade xee
!pip install -U geemap

In [ ]:
import ee
import geemap
import os
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
ee.Authenticate()
ee.Initialize(
    project= 'roads-and-highways',
    opt_url= 'https://earthengine-highvolume.googleapis.com'
)

In [ ]:
Map=geemap.Map()
Map

In [ ]:
roi= Map.draw_last_feature.geometry()

In [ ]:
time_start= ee.Date('2000')
time_end= ee.Date('2018')
time_diff= time_end.difference(time_start, 'month').round()
time_list= ee.List.sequence(0, time_diff).map(lambda t: time_start.advance(t, 'month'))
time_list

In [ ]:
flooded= (
    ee.ImageCollection("GLOBAL_FLOOD_DB/MODIS_EVENTS/V1")
    .select('flooded')
    .filterDate(time_start, time_end)
)


permanent_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').gt(60)

# flood_mask= flooded.map(lambda img:img.updateMask(permanent_water.Not()))



In [ ]:
climate_bands = ['aet', 'def', 'pdsi', 'pet', 'pr', 'ro', 'soil', 'srad', 'swe', 'tmmn', 'tmmx', 'vap', 'vpd', 'vs']
climate= (
    ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
    .filterDate(time_start, time_end)
    .select(climate_bands)
)

In [ ]:
all_bands= climate.first().bandNames().getInfo() + flooded.first().bandNames().getInfo()
all_bands

In [ ]:
all_bands= ['aet', 'def', 'pdsi', 'pet', 'pr', 'ro', 'soil', 'srad', 'swe', 'tmmn', 'tmmx', 'vap', 'vpd', 'vs','flooded']

def create_homogenized_monthly_image(date_millis):
  start_date = ee.Date(date_millis)
  end_date = start_date.advance(1, 'month')

  # Create a template image with all expected bands filled with masked zeros, and explicitly cast to float32
  template_bands_list = []
  for b in all_bands:
    template_bands_list.append(ee.Image.constant(0).updateMask(ee.Image.constant(0)).rename(b).toFloat())
  template_image = ee.Image.cat(template_bands_list)

  # Retrieve raw monthly images, already pre-selected globally
  # And cast them to float32 upon retrieval
  flood_monthly = flooded.filterDate(start_date, end_date).sum().toFloat()
  climate_monthly = climate.filterDate(start_date, end_date).median().toFloat()

  flood_bands= ['flooded']
  climate_bands= ['aet', 'def', 'pdsi', 'pet', 'pr', 'ro', 'soil', 'srad', 'swe', 'tmmn', 'tmmx', 'vap', 'vpd', 'vs']

    # Use ee.Algorithms.If to handle potential empty collections for dynamic data
  flooded_data = ee.Image(ee.Algorithms.If(
      flood_monthly.bandNames().size().gt(0),
      flood_monthly.select(flood_bands),
      template_image.select(flood_bands)
    ))
  climate_data= ee.Image(ee.Algorithms.If(
      climate_monthly.bandNames().size().gt(0),
      climate_monthly.select(climate_bands),
      template_image.select(climate_bands)
    ))
    # Combine all data into the template image, overwriting masked zeros with actual data
  combined_image= (
      template_image
      .addBands(flooded_data, None, True)
      .addBands(climate_data, None, True)
    )
  final_image= combined_image.select(all_bands).toFloat()
  return final_image.set('system:time_start', start_date.millis())

In [ ]:
collection= ee.ImageCollection(time_list.map(create_homogenized_monthly_image))
collection

In [ ]:
lc= (
    ee.ImageCollection("MODIS/061/MCD12Q1")
    .filterDate('2000', '2018')
    .select('LC_Type1')
    .mode()
)
surface_water= (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select('occurrence')
)
dem= (
    ee.Image("NASA/NASADEM_HGT/001")
    .select('elevation')
)

In [ ]:
collection=collection.map(
    lambda img:img.addBands(lc).addBands(surface_water).addBands(dem).updateMask(permanent_water.Not())

)
collection

In [ ]:
ds= xr.open_dataset(
    collection,
    engine= 'ee',
    crs= 'epsg:4326',
    geometry= roi,
    scale= 0.1
)


In [ ]:
ds= ds.sortby('time') * 1
ds

In [ ]:
ds1= ds.sel(time=slice('2015-01-01', '2017-12-01'))
ds1

In [ ]:
df= ds1.to_dataframe()
df

In [ ]:
df.describe()


In [ ]:
df.info()

In [ ]:
df.dropna(inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from sklearn.linear_model import LogisticRegression, Perceptron, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import time

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import time

import tensorflow as tf
from tensorflow.keras.layers import Dense, Activation
# Dense Neural Network
from tensorflow.keras.layers import Dense, Dropout
# Sequential Connection with Neural Network
from tensorflow.keras.models import Sequential
# Optimizers for Regression Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, PReLU

#from tensorflow.keras.layers import PReLU, ELU, Activation
#from tensorflow.keras.layers import Dense, LeakyReLU
from keras.layers import Dense, Activation, LeakyReLU, PReLU, ELU

In [ ]:
X = df.drop(columns=['flooded'], axis=1)
y = df['flooded']

In [ ]:
# Split the dataset into training, testing, and validation sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
X_train.shape[1]

In [ ]:
def build_model():
    # Sequential Neural Network - Feedforward Neural Network
    model = Sequential()
    # Units = Number of Neurons(2 * pow(n)) , Input Shape = Num of Features
    model.add(Dense(units = 64, activation = 'relu', input_shape = [len(X.keys())]))

    # Hidden Layer - I
    model.add(Dense(units = 128, activation = 'relu'))

    # Hidden Layer - II
    model.add(Dense(units = 128, activation = 'relu'))

    # Output Layer - For binary Classification
    model.add(Dense(units = 1, activation = 'sigmoid'))

    # Optimizers (alpha)
    optimizers = Adam(learning_rate = 0.001)

    # Model Compiler
    # Error Function for binary classification = 'binary_crossentropy'
    # Metrics = Metrics of Model / Check the performance of model is Accuracy
    model.compile(loss = 'binary_crossentropy', optimizer = optimizers, metrics = ['accuracy'])

    return model

In [ ]:
model1 = build_model()

In [ ]:
model1.summary()

In [ ]:
history1= model1.fit(X_train, y_train, epochs= 100, batch_size=32, validation_data=(X_val, y_val))


In [ ]:
pd.DataFrame(history1.history)[['accuracy', 'val_accuracy']].plot()

In [ ]:
fpr = {}
tpr = {}
roc_auc = {}

# Get predicted probabilities for the positive class
y_test_prob = model1.predict(X_test)

# For binary classification, roc_curve expects 1D arrays
fpr['micro'], tpr['micro'], _ = roc_curve(y_test, y_test_prob)
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

plt.figure(figsize=(8,5))
plt.plot(fpr['micro'], tpr['micro'], label= 'ROC curve (area = {:.2f})'.format(roc_auc['micro']))

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristics (ROC)')
plt.legend(loc='lower right')
plt.show()